In [81]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

In [82]:
def engineer_features(df):
    """Extract meaningful features from raw Titanic data."""
    df = df.copy()

    # --- Title from Name (one of the strongest signals) ---
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    rare = [
        "Lady",
        "Countess",
        "Capt",
        "Col",
        "Don",
        "Dr",
        "Major",
        "Rev",
        "Sir",
        "Jonkheer",
        "Dona",
    ]
    df["Title"] = df["Title"].replace(rare, "Rare")
    df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})

    # --- Family features ---
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # --- Deck from Cabin (U = unknown) ---
    df["Deck"] = df["Cabin"].str[0].fillna("U")

    # --- Fill missing values ---
    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

    # --- Log-transform Fare to reduce right skew ---
    df["Fare"] = np.log1p(df["Fare"])

    # --- Encode Sex ---
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

    # --- Drop columns we no longer need ---
    df.drop(
        ["PassengerId", "Name", "Ticket", "Cabin", "SibSp", "Parch"],
        axis=1,
        inplace=True,
    )

    # --- One-hot encode categoricals ---
    df = pd.get_dummies(
        df, columns=["Embarked", "Pclass", "Title", "Deck"], drop_first=False
    )

    # --- Cast bools to int ---
    bool_cols = df.select_dtypes(include="bool").columns
    df[bool_cols] = df[bool_cols].astype(int)

    return df

In [83]:
# --- Load & engineer features on the full dataset first ---
raw_df = pd.read_csv("./data/titanic/train.csv")
full_df = engineer_features(raw_df)

# --- Stratified split (preserves class balance) ---
train_df, val_df = train_test_split(
    full_df, test_size=0.2, random_state=42, stratify=full_df["Survived"]
)

# --- Align columns (in case a rare category only appears in one split) ---
val_df = val_df.reindex(columns=train_df.columns, fill_value=0)

# --- Normalize continuous features using TRAINING statistics only ---
continuous_cols = ["Age", "Fare", "FamilySize"]
train_stats = train_df[continuous_cols].agg(["mean", "std"])

for col in continuous_cols:
    mean, std = train_stats[col]["mean"], train_stats[col]["std"]
    train_df = train_df.copy()
    val_df = val_df.copy()
    train_df[col] = (train_df[col] - mean) / std
    val_df[col] = (val_df[col] - mean) / std

X_train = train_df.drop("Survived", axis=1).values.astype(np.float32)
y_train = train_df["Survived"].values.astype(np.float32)
X_val = val_df.drop("Survived", axis=1).values.astype(np.float32)
y_val = val_df["Survived"].values.astype(np.float32)

print(f"Features: {train_df.drop('Survived', axis=1).columns.tolist()}")
print(
    f"Train size: {len(X_train)}, Val size: {len(X_val)}, Features: {X_train.shape[1]}"
)

Features: ['Sex', 'Age', 'Fare', 'FamilySize', 'IsAlone', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Pclass_1', 'Pclass_2', 'Pclass_3', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Deck_U']
Train size: 712, Val size: 179, Features: 25


In [ ]:
n_features = X_train.shape[1]
print(f"Number of features: {n_features}")
# Larger network with BatchNorm + Dropout for regularization
# No sigmoid at the end — use BCEWithLogitsLoss (numerically more stable)
model = torch.nn.Sequential(
    torch.nn.Linear(n_features, 64),
    torch.nn.BatchNorm1d(64),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(64, 32),
    torch.nn.BatchNorm1d(32),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(32, 1),
)

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = torch.nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=300, gamma=0.5)

X_val_t = torch.from_numpy(X_val)
y_val_t = torch.from_numpy(y_val)

n_epochs = 500
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch).squeeze(1), y_batch)
        loss.backward()
        optimizer.step()
    scheduler.step()

    if (epoch + 1) % 300 == 0:
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t).squeeze(1)
            val_loss = loss_fn(val_logits, y_val_t).item()
            val_acc = ((val_logits > 0).float() == y_val_t).float().mean().item()
        print(f"Epoch {epoch + 1:4d} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

Number of features: 25
Epoch  300 | val_loss=0.5255 | val_acc=0.7654


In [85]:
model.eval()
with torch.no_grad():
    val_logits = model(X_val_t).squeeze(1)
    y_pred = (val_logits > 0).float()
    accuracy = (y_pred == y_val_t).float().mean().item()

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params}")
print(f"Final validation accuracy: {accuracy:.4f}")

Parameters: 3969
Final validation accuracy: 0.7709
